# Week 1 - API Setup & Testing

### Objective:
This notebook verifies the connectivity and functionality of the APIs required for data collection later in the project.

The following APIs are tested:
- Yahoo Finance
- FRED
- Alpha Vantage

The retrieved data will later be used for option pricing model development and empirical analysis.

## Required Packages

In [15]:
import pandas as pd
import requests
import yfinance as yf

print("Libraries imported successfully.")

Libraries imported successfully.


In [16]:
from pathlib import Path
import os

from dotenv import load_dotenv

project_root = Path.cwd().parent
load_dotenv(project_root / ".env")

fred_api_key = os.getenv("FRED_API_KEY")
alpha_vantage_api_key = os.getenv("ALPHA_VANTAGE_API_KEY")

print("FRED key loaded:", fred_api_key is not None)
print("Alpha Vantage key loaded:", alpha_vantage_api_key is not None)

FRED key loaded: True
Alpha Vantage key loaded: True


# 1. Yahoo Finance API
In this project, Yahoo Finance will be used to retrieve:
- JPM historical stock prices
- VIX historical index values

In [17]:
jpm = yf.download(
    "JPM",
    start="2018-01-01",
    end="2024-12-31"
)

jpm.head()
jpm.info()
jpm.shape

[*********************100%***********************]  1 of 1 completed

<class 'pandas.DataFrame'>
DatetimeIndex: 1760 entries, 2018-01-02 to 2024-12-30
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   (Close, JPM)   1760 non-null   float64
 1   (High, JPM)    1760 non-null   float64
 2   (Low, JPM)     1760 non-null   float64
 3   (Open, JPM)    1760 non-null   float64
 4   (Volume, JPM)  1760 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 82.5 KB


(1760, 5)

In [18]:
vix = yf.download(
    "^VIX",
    start="2018-01-01",
    end="2024-12-31"
)

vix.head()
vix.info()
vix.shape

[*********************100%***********************]  1 of 1 completed

<class 'pandas.DataFrame'>
DatetimeIndex: 1760 entries, 2018-01-02 to 2024-12-30
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   (Close, ^VIX)   1760 non-null   float64
 1   (High, ^VIX)    1760 non-null   float64
 2   (Low, ^VIX)     1760 non-null   float64
 3   (Open, ^VIX)    1760 non-null   float64
 4   (Volume, ^VIX)  1760 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 82.5 KB


(1760, 5)

Result:

The Yahoo Finance API successfully returned historical market data,
confirming that the API connection is functioning properly.

# 2. FRED API
FRED provides official U.S. macroeconomic data
For this project,
Treasury Yield will be collected as the proxy for the risk-free interest rate.

In [19]:
from fredapi import Fred

fred = Fred(api_key=fred_api_key)

treasury = fred.get_series(
    "DGS10",
    observation_start="2018-01-01",
    observation_end="2024-12-31"
)

treasury.head()
treasury.info()
treasury.shape

<class 'pandas.Series'>
DatetimeIndex: 1827 entries, 2018-01-01 to 2024-12-31
Series name: None
Non-Null Count  Dtype  
--------------  -----  
1750 non-null   float64
dtypes: float64(1)
memory usage: 28.5 KB


(1827,)

# 3. Alpha Vantage API
Alpha Vantage provides financial market data through REST APIs.
This section performs a connection test using the provider's 100-row compact daily response. It is a secondary API check; the 2018–2024 JPM series used downstream comes from Yahoo Finance.

In [20]:
import requests

url = (
    "https://www.alphavantage.co/query?"
    "function=TIME_SERIES_DAILY"
    "&symbol=JPM"
    "&outputsize=compact"  # latest 100 observations: connection test only
    f"&apikey={alpha_vantage_api_key}"
)

response = requests.get(url, timeout=30)
response.raise_for_status()
data = response.json()
if "Time Series (Daily)" not in data:
    raise RuntimeError(f"Alpha Vantage response did not contain daily data: {list(data)}")

alpha_jpm = pd.DataFrame.from_dict(data["Time Series (Daily)"], orient="index")
alpha_jpm.index = pd.to_datetime(alpha_jpm.index)
alpha_jpm.index.name = "Date"
print(response.status_code)

200


# 4. Data Pull
Save the data to local

In [21]:
jpm.to_csv("raw_data/JPM_Yahoo_2018_2024.csv")
vix.to_csv("raw_data/VIX_2018_2024.csv")
treasury.to_csv("raw_data/Treasury_2018_2024.csv")
alpha_jpm.to_csv("raw_data/JPM_AlphaVantage_compact_2026.csv")

# Summary

All Week 1 objectives have been completed.

Completed tasks include:

- Defined project data requirements.
- Configured and tested Yahoo Finance, FRED, and Alpha Vantage APIs.
- Retrieved the 2018–2024 JPM, VIX, and U.S. Treasury series used by the project.
- Retained the Alpha Vantage compact response as a clearly named 100-row connection-test file; it is not used in the downstream 2018–2024 pipeline.
- Exported the raw datasets for preprocessing and model development.

The project environment and data collection pipeline provide the foundation for the later preprocessing and chooser-option modelling stages.